# Projeto 1: Limpeza dataset ecommerce

## Importação das bibliotecas


In [18]:
import pandas as pd
import numpy as np

## Importação dos dados

In [19]:

importacao=pd.read_csv('vendas_ecommerce_dados_brutos.csv')
df=importacao.copy()

## Entendendo os dados


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 504 entries, 0 to 503
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_pedido          504 non-null    int64 
 1   data_pedido        501 non-null    object
 2   id_cliente         501 non-null    object
 3   nome_cliente       481 non-null    object
 4   email              483 non-null    object
 5   telefone           444 non-null    object
 6   idade              481 non-null    object
 7   cidade             486 non-null    object
 8   estado             492 non-null    object
 9   categoria_produto  500 non-null    object
 10  produto            502 non-null    object
 11  quantidade         490 non-null    object
 12  preco_unitario     490 non-null    object
 13  valor_total        472 non-null    object
 14  forma_pagamento    493 non-null    object
 15  status_pedido      501 non-null    object
 16  avaliacao_cliente  418 non-null    object
dt

In [21]:
df.describe()

,id_pedido
count,504.000000
mean,1243.095238
std,140.767305
min,1001.000000
25%,1122.750000
50%,1241.500000
75%,1361.250000
max,1583.000000


## Tratando colunas


In [22]:
#Tratando colunas numéricas float

#Limpando as colunas e mantendo somente os dados numéricos e os separadores de decimais e de milhar
for col in ['valor_total','preco_unitario']:
    df[col]=df[col].astype('str').str.replace(r'[^\d\.\,]','',regex=True).str.strip()

#lidando com as colunas numéricas float despadronizadas, algumas usavam ',' como separador de decimal e outras usavam 
# '.' oq dificultava em criar uma solução geral para a coluna 
for col in ['valor_total','preco_unitario']:
    virgula= df[col].str.contains(r',\d{2}$',regex=True)
    df.loc[virgula,col] = (df.loc[virgula,col].str.replace('.','')
                            .str.replace(',','.')
                            .str.strip())
    
#filtro=df['preco_unitario'].astype('str').str.contains(r'^\s$',regex=True)
#df.loc[filtro,'preco_unitario'] = np.nan   # não funcionou


for col in ['valor_total','preco_unitario']:
    df.loc[df[col]=='', col] = np.nan
    df[col]=df[col].astype('float')


In [23]:
#colunas numericas int

# essas linhas servem para lidar com os caracteres que não são dígitos
df['avaliacao_cliente']=df['avaliacao_cliente'].str.replace(r'\D','',regex=True).str.strip()
df['quantidade']=df['quantidade'].str.replace(r'\D','',regex=True).str.strip()
df['quantidade'].value_counts()


quantidade
1    173
2    124
3     80
4     57
5     41
0      9
       6
Name: count, dtype: int64

In [24]:
#trocando os espaços por nulos para poder criar colunas numéricas e imputar valores aos nulos
for col in ['quantidade','avaliacao_cliente']:
   df.loc[df[col]=='',col] = np.nan

# estava dando erro ao tentar converter diretamente para int por conta dos nulos, coloquei para float somente para realizar operações de mediana e tratar os nulos
df['quantidade']=df['quantidade'].astype('float')
df['avaliacao_cliente']=df['avaliacao_cliente'].astype('float')

#imputando a mediana aos nulos
for col in ['quantidade','valor_total','preco_unitario','avaliacao_cliente']:
   df[col]=df[col].fillna((df[col].median(skipna=True)))

#conversão para int
df['quantidade']=df['quantidade'].astype('Int64')
df['avaliacao_cliente']=df['avaliacao_cliente'].astype('Int64')

In [25]:

df['id_cliente']=df['id_cliente'].astype('str').str.replace(r'\D','',regex=True).str.strip()
df.loc[df['id_cliente']=='','id_cliente'] = np.nan

# Exclusão das linhas com id nulo. 
# mesmo que estivesse com todas as outras colunas preenchidas sem id não é possível diferenciar/localizar entre os cadastros
df=df.dropna(subset='id_cliente')
df['id_pedido']=df['id_pedido'].astype('int')
df['id_cliente']=df['id_cliente'].astype('int')

In [26]:
#retirando não dígitos
df['idade']=df['idade'].str.replace(r'\D','',regex=True)
df['idade']=df['idade'].str.strip()
df.loc[df['idade']=='','idade'] = np.nan

# O int padrão python/numpy não aceita nulos, diferente do Int do pandas (o float padrão aceita também).
df['idade']=df['idade'].astype('Int64')


In [27]:
#imputando nulos para a coluna de idade
# essas condição normalmente iria depender das regras de negócio, então
df.loc[df['idade']<18,'idade'] = df['idade'].median()
df['idade']=df['idade'].fillna((df['idade'].median()))
df.loc[df['idade']>80,'idade'] = df['idade'].median()


In [28]:
#trocando os separadores das datas
df['data_pedido']=df['data_pedido'].str.replace('-','/')

#formatando a coluna para o tipo data
df['data_pedido']=pd.to_datetime(df['data_pedido'],format='mixed')

In [29]:
#lidando com caracteres indesejados 
df['nome_cliente']=df['nome_cliente'].str.replace(r'[^\w\s\.]','',regex=True).str.title().str.strip()
df.loc[df['nome_cliente']=='','nome_cliente'] = np.nan

df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 0 to 503
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_pedido          500 non-null    int64         
 1   data_pedido        500 non-null    datetime64[ns]
 2   id_cliente         500 non-null    int64         
 3   nome_cliente       474 non-null    object        
 4   email              481 non-null    object        
 5   telefone           444 non-null    object        
 6   idade              500 non-null    Int64         
 7   cidade             484 non-null    object        
 8   estado             491 non-null    object        
 9   categoria_produto  500 non-null    object        
 10  produto            500 non-null    object        
 11  quantidade         500 non-null    Int64         
 12  preco_unitario     500 non-null    float64       
 13  valor_total        500 non-null    float64       
 14  forma_pagamento

In [30]:

df['telefone']=df['telefone'].str.replace(r'[^\d\-]','',regex=True).str.replace(r'^55','',regex=True)
df['telefone']=df['telefone'].replace('-','') #sem o str o python n analisa os caracteres individualmente, mas sim o valor da célula como um todo
df.loc[df['telefone']=='','telefone'] = np.nan

for col in ['forma_pagamento','status_pedido','categoria_produto','cidade']:
    df[col]=df[col].astype('str').str.strip().str.title()
    df[col]=df[col].str.replace(r'[^\w\s]','',regex=True)
    df.loc[df[col]=='',col] = np.nan
    
df['produto']=df['produto'].astype('str').str.strip().str.title()
df['produto']=df['produto'].str.replace(r'[^\w\s]','',regex=True)
df.loc[df['produto']=='','produto'] = np.nan




In [31]:
df['email']=df['email'].str.replace(r'[^\w\.\+\-\@]','',regex=True).str.lower()
df.loc[df['email']=='','email'] = np.nan

In [32]:
#padronizando os registros de estado
df['estado'] = df['estado'].str.upper()

mapa_estados = {
    'MINAS GERAIS': 'MG',
    'CEARÁ': 'CE', 'CEARA': 'CE',
    'BAHIA': 'BA',
    'PERNAMBUCO': 'PE',
    'RIO GRANDE DO SUL': 'RS',
    'AMAZONAS': 'AM',
    'RIO DE JANEIRO': 'RJ',
    'PARANÁ': 'PR', 'PARANA': 'PR',
    'SANTA CATARINA': 'SC',
    'SÃO PAULO': 'SP', 'SAO PAULO': 'SP',
    'SERGIPE': 'SE',
    'DISTRITO FEDERAL': 'DF',
    'NÃO INFORMADO': np.nan, 'NAO INFORMADO': np.nan, '?': np.nan,
}

df['estado'] = df['estado'].replace(mapa_estados)
df

,id_pedido,data_pedido,id_cliente,nome_cliente,email,telefone,idade,cidade,estado,categoria_produto,produto,quantidade,preco_unitario,valor_total,forma_pagamento,status_pedido,avaliacao_cliente
0,1041,2023-07-26,306,Renan Ribeiro,leonardo88@example.org,NaN,59,Florianópolis,SC,Roupas,Camiseta Básica,2,49.9,99.8,Cartão De Débito,Devolvido,5
1,1047,2024-03-27,195,Diego Melo,novaisdavi@example.com,1197309-1356,70,Contagem,MG,Roupas,Calça Jeans,4,129.9,519.6,Cartão De Débito,Processando,10
2,1344,2023-10-16,165,Nicole Moreira,liviacaldeira@example.com,94400-6566,49,Porto Alegre,RS,Esporte E Lazer,Bola De Futebol Oficial,4,89.9,359.6,Boleto Bancário,Cancelado,1
3,1137,2025-08-01,45,Fernanda Freitas,ucorreia@example.com,NaN,41,Olinda,PE,Roupas,Vestido Casual,3,159.9,479.7,Boleto Bancário,Em Transporte,1
4,1110,2023-08-27,149,Cecilia Novaes,ryanborges@example.org,8594156-8143,29,Petrópolis,RJ,Roupas,Calça Jeans,2,129.9,259.8,Cartão De Crédito,Cancelado,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,1007,2024-12-25,109,Esther Freitas,novaesdavi-lucas@example.net,8593142-9713,67,Santos,SP,Roupa,Jaqueta CortaVento,3,199.9,599.7,Cartão De Débito,Devolvido,4
500,1221,2024-03-17,265,Luiz Felipe Melo,nascimentopedro@example.net,1197344-8413,59,Nan,RS,Eletrônicos,Notebook,2,3299.0,6598.0,Boleto Bancário,Devolvido,5
501,1462,2025-08-22,139,Davi Miguel Freitas,guerrarenan@example.org,7999027-9526,36,Duque De Caxias,RJ,Roupas,Vestido Casual,2,159.9,319.8,Pix,Cancelado,5
502,1300,2024-02-24,319,NaN,ravi-luccadas-neves@example.net,7196635-4587,45,Parintins,AM,Casa E Decoração,Tapete Sala,4,249.9,999.6,Pix,Processando,4


In [33]:
df=df.fillna('Não Informado')
df['telefone']=='-'
df

,id_pedido,data_pedido,id_cliente,nome_cliente,email,telefone,idade,cidade,estado,categoria_produto,produto,quantidade,preco_unitario,valor_total,forma_pagamento,status_pedido,avaliacao_cliente
0,1041,2023-07-26,306,Renan Ribeiro,leonardo88@example.org,Não Informado,59,Florianópolis,SC,Roupas,Camiseta Básica,2,49.9,99.8,Cartão De Débito,Devolvido,5
1,1047,2024-03-27,195,Diego Melo,novaisdavi@example.com,1197309-1356,70,Contagem,MG,Roupas,Calça Jeans,4,129.9,519.6,Cartão De Débito,Processando,10
2,1344,2023-10-16,165,Nicole Moreira,liviacaldeira@example.com,94400-6566,49,Porto Alegre,RS,Esporte E Lazer,Bola De Futebol Oficial,4,89.9,359.6,Boleto Bancário,Cancelado,1
3,1137,2025-08-01,45,Fernanda Freitas,ucorreia@example.com,Não Informado,41,Olinda,PE,Roupas,Vestido Casual,3,159.9,479.7,Boleto Bancário,Em Transporte,1
4,1110,2023-08-27,149,Cecilia Novaes,ryanborges@example.org,8594156-8143,29,Petrópolis,RJ,Roupas,Calça Jeans,2,129.9,259.8,Cartão De Crédito,Cancelado,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,1007,2024-12-25,109,Esther Freitas,novaesdavi-lucas@example.net,8593142-9713,67,Santos,SP,Roupa,Jaqueta CortaVento,3,199.9,599.7,Cartão De Débito,Devolvido,4
500,1221,2024-03-17,265,Luiz Felipe Melo,nascimentopedro@example.net,1197344-8413,59,Nan,RS,Eletrônicos,Notebook,2,3299.0,6598.0,Boleto Bancário,Devolvido,5
501,1462,2025-08-22,139,Davi Miguel Freitas,guerrarenan@example.org,7999027-9526,36,Duque De Caxias,RJ,Roupas,Vestido Casual,2,159.9,319.8,Pix,Cancelado,5
502,1300,2024-02-24,319,Não Informado,ravi-luccadas-neves@example.net,7196635-4587,45,Parintins,AM,Casa E Decoração,Tapete Sala,4,249.9,999.6,Pix,Processando,4


## Exportação


In [34]:
df.to_csv('ecommerce_limpo.csv', index=False)